# Demo de Stable Retro: Airstriker-Genesis

Este notebook utiliza **Stable Retro** para entrenar agentes de Reinforcement Learning.

## 1. Importar dependencias

In [ ]:
import retro
import numpy as np
import matplotlib.pyplot as plt
from IPython import display
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
import gymnasium as gym

## 2. Configuración del Entorno

In [ ]:
game_name = 'Airstriker-Genesis-v0'

if game_name not in retro.data.list_games():
    print(f"ERROR: No se encontró el ROM de {game_name}.")
else:
    print(f"{game_name} encontrado.")

env = retro.make(game=game_name, render_mode='rgb_array')
print(f"Action Space: {env.action_space}")
print(f"Observation Space: {env.observation_space.shape}")

## 3. Demo con Movimientos Aleatorios

In [ ]:
obs, info = env.reset()
plt.figure(figsize=(8,6))

frame = env.render()
img = plt.imshow(frame)
plt.axis('off')

for _ in range(100):
    display.clear_output(wait=True)
    
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    
    img.set_data(env.render())
    display.display(plt.gcf())
    
    if terminated or truncated:
        obs, info = env.reset()

env.close()

## 4. Entrenamiento con Stable Baselines3

In [ ]:
def make_env():
    env = retro.make(game=game_name, render_mode='rgb_array')
    return env

env_vec = DummyVecEnv([make_env])
env_vec = VecFrameStack(env_vec, n_stack=4)

In [ ]:
model = PPO("CnnPolicy", env_vec, verbose=1)
print("Entrenando 5000 pasos...")
model.learn(total_timesteps=5000)
model.save("ppo_airstriker")

## 5. Test del Agente Entrenado

In [ ]:
model = PPO.load("ppo_airstriker")

obs = env_vec.reset()
plt.figure(figsize=(8,6))
# En entornos vectorizados, usamos get_images() o el primer frame
img = plt.imshow(env_vec.get_images()[0])
plt.axis('off')

for _ in range(300):
    display.clear_output(wait=True)
    
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = env_vec.step(action)
    
    img.set_data(env_vec.get_images()[0])
    display.display(plt.gcf())
    
    if done.any():
        obs = env_vec.reset()

env_vec.close()